# CORUM Validation — Leakage-Fixed Model (4 clusters)

In [ ]:
!pip install requests pandas numpy scipy matplotlib -q

In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import fisher_exact
import matplotlib.pyplot as plt
import json

from google.colab import drive
drive.mount('/content/drive')
DRIVE_PATH = '/content/drive/MyDrive/'

In [ ]:
# Load gene names
with pd.HDFStore(DRIVE_PATH + 'gene_network_data.h5') as store:
    tec = store['TEC']
gene_names = tec.index.tolist()
n_genes = len(gene_names)
gene_to_idx = {g: i for i, g in enumerate(gene_names)}
del tec
print(f'Loaded {n_genes} genes')

In [ ]:
# Load leakage-fixed embeddings and cluster labels
embeddings_np  = np.load(DRIVE_PATH + 'noleak_gnn_embeddings.npy')
cluster_labels = np.load(DRIVE_PATH + 'noleak_gnn_cluster_labels.npy')
print(f'Embeddings: {embeddings_np.shape}')
print(f'Cluster sizes: {np.bincount(cluster_labels)}')

In [ ]:
# Load CORUM 5.0
corum_df = pd.read_csv(DRIVE_PATH + 'corum_allComplexes.txt', sep='\t')
org_col = 'organism' if 'organism' in corum_df.columns else 'Organism'
corum_df = corum_df[corum_df[org_col] == 'Human']

subunit_col = [c for c in corum_df.columns if 'subunit' in c.lower() and 'gene' in c.lower()]
name_col    = [c for c in corum_df.columns if 'complex' in c.lower() and 'name' in c.lower()]
subunit_col = subunit_col[0] if subunit_col else 'subunits(Gene name)'
name_col    = name_col[0]    if name_col    else 'ComplexName'

complexes = {}
for _, row in corum_df.iterrows():
    subunits = [g.strip() for g in str(row[subunit_col]).split(';') if g.strip()]
    if len(subunits) >= 2:
        complexes[row[name_col]] = set(subunits)

all_corum_genes = set(g for genes in complexes.values() for g in genes)
print(f'CORUM complexes: {len(complexes)}')
print(f'Our genes in CORUM: {len(set(gene_names) & all_corum_genes)}')

In [ ]:
# Per-complex enrichment with Bonferroni correction
enrichment_results = []
n_clusters = len(np.unique(cluster_labels))

for complex_name, complex_genes in complexes.items():
    overlap_genes = [g for g in complex_genes if g in gene_to_idx]
    if len(overlap_genes) < 3:
        continue
    overlap_indices  = [gene_to_idx[g] for g in overlap_genes]
    overlap_clusters = cluster_labels[overlap_indices]

    for cluster_id in range(n_clusters):
        in_c     = np.sum(overlap_clusters == cluster_id)
        not_in_c = len(overlap_genes) - in_c
        total_in_c     = np.sum(cluster_labels == cluster_id)
        total_not_in_c = n_genes - total_in_c
        contingency = [[in_c, not_in_c],
                       [total_in_c - in_c, total_not_in_c - not_in_c]]
        try:
            odds_ratio, p_value = fisher_exact(contingency, alternative='greater')
            if p_value < 0.05:
                enrichment_results.append({
                    'complex': complex_name,
                    'cluster': cluster_id,
                    'complex_size': len(complex_genes),
                    'overlap': in_c,
                    'odds_ratio': odds_ratio,
                    'p_value': p_value
                })
        except:
            pass

enrichment_df = pd.DataFrame(enrichment_results).sort_values('p_value') if enrichment_results else pd.DataFrame()
# Bonferroni correction
n_tests = len(complexes) * n_clusters
if not enrichment_df.empty:
    enrichment_df['p_bonferroni'] = enrichment_df['p_value'] * n_tests
    sig_df = enrichment_df[enrichment_df['p_bonferroni'] < 0.05]
    print(f'Significant complexes (Bonferroni): {len(sig_df)}')
    print(sig_df.head(20).to_string(index=False))

In [ ]:
# Co-complex pair co-assignment rate
co_complex_pairs = set()
for complex_genes in complexes.values():
    genes_in_data = [g for g in complex_genes if g in gene_to_idx]
    for i in range(len(genes_in_data)):
        for j in range(i+1, len(genes_in_data)):
            g1, g2 = genes_in_data[i], genes_in_data[j]
            co_complex_pairs.add((min(g1, g2), max(g1, g2)))

print(f'Co-complex pairs: {len(co_complex_pairs)}')

same, diff = 0, 0
for g1, g2 in list(co_complex_pairs)[:10000]:
    if cluster_labels[gene_to_idx[g1]] == cluster_labels[gene_to_idx[g2]]:
        same += 1
    else:
        diff += 1

total = same + diff
same_rate = same / total
expected  = sum((np.sum(cluster_labels == c) / n_genes)**2 for c in np.unique(cluster_labels))

print(f'\n=== CO-COMPLEX PAIR ANALYSIS ===')
print(f'Same cluster: {same}/{total} ({same_rate*100:.1f}%)')
print(f'Expected by chance: {expected*100:.1f}%')
print(f'Enrichment: {same_rate/expected:.2f}x')

In [ ]:
# Plot
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].bar(['GNN Modules', 'Random Expected'],
            [same_rate * 100, expected * 100],
            color=['#4CAF50', '#FF9800'], width=0.5)
axes[0].set_ylabel('% Co-complex pairs in same cluster', fontsize=12)
axes[0].set_title('GNN Module Coherence vs Random\n(CORUM Protein Complexes)', fontsize=12)
for i, v in enumerate([same_rate * 100, expected * 100]):
    axes[0].text(i, v + 0.5, f'{v:.1f}%', ha='center', fontsize=12, fontweight='bold')

if not enrichment_df.empty:
    top10 = sig_df.head(10)
    axes[1].barh(range(len(top10)), -np.log10(top10['p_bonferroni'].clip(lower=1e-300)), color='#2196F3')
    axes[1].set_yticks(range(len(top10)))
    axes[1].set_yticklabels([c[:35] for c in top10['complex']], fontsize=8)
    axes[1].set_xlabel('-log10(p Bonferroni)', fontsize=12)
    axes[1].set_title('Top CORUM Complexes Enriched\nin GNN Modules', fontsize=12)

plt.tight_layout()
plt.savefig('noleak_corum_validation_results.png', dpi=300, bbox_inches='tight')
plt.show()
print('Saved: noleak_corum_validation_results.png')

In [ ]:
# Save results
results = {
    'leakage_fixed': True,
    'co_complex_same_cluster_rate': same_rate,
    'random_expected_rate': expected,
    'enrichment_over_random': same_rate / expected,
    'significant_complexes_bonferroni': len(sig_df) if not enrichment_df.empty else 0,
    'top_complexes': sig_df.head(20).to_dict('records') if not enrichment_df.empty else []
}
with open('noleak_corum_validation_results.json', 'w') as f:
    json.dump(results, f, indent=2, default=str)
print('Saved: noleak_corum_validation_results.json')